# Day 17. Feed Forward Network (FFN)

**오늘의 목표**
- Attention과 FFN의 차이를 한 문장으로 설명할 수 있다: **Attention은 Token끼리 섞고, FFN은 Token마다 따로 처리한다**
- FFN이 Token마다 독립적으로 동작한다는 것(position-wise)을 **직접 실험으로 증명**한다
- 왜 4배로 늘렸다가 줄이는지, "패턴 감지 → 정보 기록"이라는 해석을 이해한다
- nanoGPT 방식의 `MLP` 클래스를 만들고, GPT 파라미터의 약 2/3가 FFN이라는 걸 계산한다
- 요즘 모델(LLaMA)이 쓰는 SwiGLU를 가볍게 본다

> Day 4에서 FFN = `Linear(×4) → GELU → Linear(÷4)`이고 GELU가 왜 꼭 필요한지 봤어요.
> 오늘은 그 FFN이 **Transformer Block 안에서 어떤 역할**을 하는지에 집중합니다.
>
> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요. `# TODO` 셀은 연습 문제입니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) + 0.0 for v in t.tolist()]   # +0.0: -0.0 표시 방지

## 1. Block 안에서 FFN의 자리

Transformer Block 하나는 크게 두 부분입니다.

```text
x  (B, T, C)
 ↓
Self-Attention   ← Token끼리 정보를 주고받는다  ("소통")
 ↓
FFN              ← 각 Token이 받은 정보를 혼자 곱씹는다  ("생각")
 ↓
x  (B, T, C)
```

Karpathy는 이걸 **"communication followed by computation"** 이라고 표현합니다.
- Attention: Token **사이**의 일 (T 축을 따라 섞음)
- FFN: Token **안**의 일 (C 축만 변환)

## 2. FFN은 Token마다 따로 동작한다 (position-wise)

### 2-1. 한 번에 vs 하나씩

`(B, T, C)` 전체를 FFN에 넣은 결과와, Token을 **하나씩 꺼내서** 넣은 결과가 같은지 봅시다.

In [ ]:
B, T, C = 2, 5, 16

ffn = nn.Sequential(
    nn.Linear(C, 4 * C),
    nn.GELU(),
    nn.Linear(4 * C, C),
)

x = torch.randn(B, T, C)

all_at_once = ffn(x)                                   # (B, T, C) 통째로

one_by_one = torch.zeros(B, T, C)
for b in range(B):
    for t in range(T):
        one_by_one[b, t] = ffn(x[b, t])                # 벡터 (C,) 하나씩

print("통째로 :", tuple(all_at_once.shape))
print("하나씩 :", tuple(one_by_one.shape))
print("같은가?", torch.allclose(all_at_once, one_by_one, atol=1e-6))

**같습니다.** FFN은 `(B, T, C)`를 받아도 실제로는 **Token 벡터 `(C,)` 하나하나에 똑같은 함수를 적용**할 뿐이에요.
Day 3에서 본 "`nn.Linear`는 마지막 축에만 작용한다"가 그대로 이어지는 이야기입니다.

### 2-2. 다른 Token을 바꾸면?

이번엔 **2번 Token만 바꿨을 때 0번 Token의 출력이 변하는지** 봅시다.
비교 대상으로 Day 12에서 만든 Q/K/V Attention(마스크 없음)을 같이 돌립니다.

In [ ]:
class SimpleAttention(nn.Module):
    # Day 12의 한 Head Attention (마스크 없음)
    def __init__(self, C):
        super().__init__()
        self.q = nn.Linear(C, C, bias=False)
        self.k = nn.Linear(C, C, bias=False)
        self.v = nn.Linear(C, C, bias=False)

    def forward(self, x):
        q, k, v = self.q(x), self.k(x), self.v(x)
        att = q @ k.transpose(-2, -1) / C ** 0.5
        return F.softmax(att, dim=-1) @ v

attn = SimpleAttention(C)

x1 = torch.randn(1, T, C)
x2 = x1.clone()
x2[0, 2] = torch.randn(C)          # 2번 Token만 다른 값으로 교체

with torch.no_grad():
    ffn_changed  = not torch.allclose(ffn(x1)[0, 0],  ffn(x2)[0, 0])
    attn_changed = not torch.allclose(attn(x1)[0, 0], attn(x2)[0, 0])

print("2번 Token을 바꿨을 때 0번 Token의 출력이 바뀌었나?")
print("  FFN       :", ffn_changed)
print("  Attention :", attn_changed)

| | 다른 Token이 바뀌면 내 출력도 바뀌나? | 섞는 축 |
|---|---|---|
| **Attention** | 바뀐다 | T 축 (Token 사이) |
| **FFN** | 안 바뀐다 | C 축 (Token 안) |

그래서 둘 다 필요합니다.
- Attention만 있으면: 정보를 모으기만 하고 **복잡한 변환(비선형 계산)** 을 할 곳이 부족해요. (Attention은 V의 가중평균이라 V 입장에서는 선형이에요)
- FFN만 있으면: 각 Token이 **문맥을 전혀 모른 채** 혼자 계산해요.

## 3. 왜 4배로 늘렸다가 줄일까?

### 3-1. "패턴 감지기 + 정보 기록기" 해석

FFN을 두 단계로 나눠 보면 이렇게 해석할 수 있습니다.

```text
h = GELU(x @ W1 + b1)     W1: (C, 4C)  → 4C개의 "패턴 감지기(뉴런)"
out = h @ W2 + b2         W2: (4C, C)  → 뉴런마다 "켜지면 더할 정보" 벡터
```

- `W1`의 **열 하나** = 감지기 하나. 입력 `x`와 Dot Product를 해서 **그 패턴이 있는지** 점수를 냅니다. (Day 2: Dot Product가 크다 = 방향이 비슷하다)
- `W2`의 **행 하나** = 그 감지기가 켜졌을 때 **출력에 더할 벡터**입니다.

(PyTorch의 `nn.Linear`는 weight를 `(출력, 입력)`으로 저장하니 코드에서는 `c_fc.weight`의 **행**, `c_proj.weight`의 **열**이 각각에 해당해요. Day 3 3-2 참고)

뉴런이 많을수록(4C) **더 많은 패턴을 기억**할 수 있습니다. 그래서 FFN을 "모델의 지식 저장소"라고 부르기도 해요.

### 3-2. 손으로 만든 뉴런 하나

벡터 길이 `C = 4`, 각 칸의 의미를 `[사람, 장소, 과거, 부정]`이라고 정해봅시다.
"**사람 + 과거**"가 함께 있을 때만 켜지는 뉴런을 만들고, 켜지면 출력의 '장소' 칸에 `+1`을 더하게 해볼게요.
(코드에서는 `c_fc.weight`의 행 = 감지기, `c_proj.weight`의 열 = 기록할 정보입니다)

In [ ]:
features = ["사람", "장소", "과거", "부정"]

c_fc   = nn.Linear(4, 2)    # 뉴런 2개 (진짜 FFN은 4C개)
c_proj = nn.Linear(2, 4)

with torch.no_grad():
    # 뉴런 0: '사람'과 '과거'가 둘 다 있어야 켜지도록 (하나만 있으면 4 - 6 = -2 → 거의 0)
    c_fc.weight[0] = torch.tensor([4.0, 0.0, 4.0, 0.0])
    c_fc.bias[0]   = -6.0
    # 뉴런 1: '장소'가 있으면 켜짐
    c_fc.weight[1] = torch.tensor([0.0, 4.0, 0.0, 0.0])
    c_fc.bias[1]   = -2.0

    # 뉴런 0이 켜지면 → '장소' 칸에 +1 을 쓴다 (예: "사람이 과거에 했다 → 어딘가 갔을 것")
    # 뉴런 1이 켜지면 → '부정' 칸에 -1 을 쓴다 (예: 임의의 정보)
    c_proj.weight.zero_()
    c_proj.weight[:, 0] = torch.tensor([0.0, 1.0, 0.0, 0.0])
    c_proj.weight[:, 1] = torch.tensor([0.0, 0.0, 0.0, -1.0])
    c_proj.bias.zero_()

inputs = {
    "사람만        ": [1.0, 0.0, 0.0, 0.0],
    "과거만        ": [0.0, 0.0, 1.0, 0.0],
    "사람 + 과거   ": [1.0, 0.0, 1.0, 0.0],
    "장소          ": [0.0, 1.0, 0.0, 0.0],
}

with torch.no_grad():
    for name, v in inputs.items():
        x_in = torch.tensor(v)
        h = F.gelu(c_fc(x_in))
        out = c_proj(h)
        print(f"{name} 뉴런 활성도 {r(h, 2)}  →  출력에 더해지는 값 {r(out, 2)}")

- "사람만", "과거만"일 때 뉴런 0은 거의 0 → 아무것도 안 씀
- "사람 + 과거"일 때만 뉴런 0이 켜지고 → `W2`에 적어둔 정보(`장소 +1` 방향)가 출력에 더해짐

진짜 모델에서는 이런 뉴런이 **Block마다 수천 개** 있고, 사람이 정하지 않고 **학습으로** 생깁니다.
(4C로 늘리는 이유 = 감지기를 많이 두기 위해, 다시 C로 줄이는 이유 = 다음 Block에 같은 모양으로 넘기기 위해)

### 3-3. 실제 FFN의 뉴런은 얼마나 켜질까?

In [ ]:
C_demo = 64
ffn_demo = nn.Sequential(nn.Linear(C_demo, 4 * C_demo), nn.GELU(), nn.Linear(4 * C_demo, C_demo))

with torch.no_grad():
    h = ffn_demo[1](ffn_demo[0](torch.randn(1000, C_demo)))   # GELU 통과 직후

plt.figure(figsize=(7, 3.5))
plt.hist(h.flatten(), bins=80, color="#4C72B0")
plt.title("GELU 통과 후 뉴런 활성도 분포 (학습 전, 랜덤 Weight)", fontsize=12)
plt.xlabel("활성도")
plt.ylabel("개수")
plt.grid(alpha=0.3)
plt.show()

print(f"0.1보다 작은 값의 비율: {(h < 0.1).float().mean().item()*100:.1f}%")

절반 넘는 뉴런이 **거의 꺼져(0 근처)** 있습니다. 입력마다 **일부 뉴런만 켜지는** 구조라서, 서로 다른 패턴을 서로 다른 뉴런이 담당할 수 있어요.

## 4. Dropout (잠깐)

nanoGPT의 FFN 끝에는 `Dropout`이 붙어 있습니다.
**학습할 때만** 일부 값을 랜덤하게 0으로 만들어서, 특정 뉴런에 너무 의존하지 않게(과적합 방지) 하는 장치예요.

In [ ]:
drop = nn.Dropout(p=0.5)
v = torch.ones(10)

drop.train()
print("train 모드:", r(drop(v), 1), " ← 절반쯤 0, 나머지는 1/(1-p)=2배로 키워 평균 유지")
drop.eval()
print("eval  모드:", r(drop(v), 1), " ← 아무것도 안 함")

> 💡 그래서 생성(Inference)할 때는 꼭 `model.eval()`을 불러야 합니다. 안 그러면 매번 랜덤하게 정보가 지워져요.
> 작은 mini GPT에서는 `dropout=0.0`으로 끄고 시작해도 괜찮습니다.

## 5. nanoGPT 방식 `MLP` 클래스

GPT 코드에서는 FFN을 보통 `MLP`라고 부릅니다. 이름(`c_fc`, `c_proj`)은 GPT-2 원본 Weight 이름을 따른 거예요.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.c_fc    = nn.Linear(n_embd, 4 * n_embd)   # C → 4C : 패턴 감지기
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * n_embd, n_embd)   # 4C → C : 정보 기록
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

mlp = MLP(n_embd=32)
x = torch.randn(2, 8, 32)
print("입력:", tuple(x.shape), "→ 출력:", tuple(mlp(x).shape))
print("파라미터:", sum(p.numel() for p in mlp.parameters()))

## 6. 파라미터 세기: GPT의 약 2/3는 FFN

bias를 빼고 Weight만 보면:

| 부분 | 구성 | Weight 개수 |
|---|---|---|
| Attention | `c_attn (C → 3C)` + `c_proj (C → C)` | `3C² + C² = 4C²` |
| FFN (MLP) | `c_fc (C → 4C)` + `c_proj (4C → C)` | `4C² + 4C² = 8C²` |

**FFN이 Attention의 2배**입니다. Block 하나의 약 2/3가 FFN이에요.
GPT-2 small(`C = 768`, 12층)로 직접 세어봅시다. (Day 14의 Attention 구조를 그대로 씀)

In [ ]:
def count(m):
    return sum(p.numel() for p in m.parameters())

class AttnShape(nn.Module):
    # 파라미터 개수만 세기 위한 Day 14 CausalSelfAttention의 Linear 구성
    def __init__(self, C):
        super().__init__()
        self.c_attn = nn.Linear(C, 3 * C)
        self.c_proj = nn.Linear(C, C)

C_gpt2, n_layer = 768, 12
attn_p = count(AttnShape(C_gpt2))
mlp_p  = count(MLP(C_gpt2))

print(f"Block 하나 - Attention : {attn_p:>12,}")
print(f"Block 하나 - FFN       : {mlp_p:>12,}")
print(f"FFN 비율               : {mlp_p / (attn_p + mlp_p) * 100:.1f}%")
print()
print(f"12층 전체 FFN          : {mlp_p * n_layer:>12,}")
print(f"12층 전체 Attention    : {attn_p * n_layer:>12,}")

GPT-2 small 전체가 약 1.24억 개인데, 그중 **약 5,700만 개가 FFN**입니다.
(나머지는 Attention 약 2,800만 + Embedding 약 3,900만 + LayerNorm 조금)

> 💡 "모델이 지식을 어디에 저장하나?"라는 질문에 FFN이 자주 지목되는 이유가 이 크기 때문이기도 해요.

## 7. 요즘 모델의 FFN: SwiGLU (가볍게)

LLaMA, Mistral, Qwen 같은 최신 모델은 FFN을 조금 바꾼 **SwiGLU**를 씁니다.

```text
GPT-2 :  out = W2( GELU(W1 x) )
SwiGLU:  out = W2( SiLU(W1 x) * (W3 x) )      ← Linear가 3개, 하나가 "문(gate)" 역할
```

- `W3 x`가 **게이트**: 뉴런마다 "얼마나 통과시킬지"를 입력에 따라 곱해서 조절
- Linear가 3개라 hidden을 4C로 두면 파라미터가 늘어나서, 보통 **hidden ≈ 8/3 · C**로 줄여 총량을 비슷하게 맞춥니다.
  (`3 × C × (8/3)C = 8C²` → GPT-2 FFN의 `8C²`와 같음)

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        hidden = int(8 / 3 * n_embd)
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(n_embd, hidden, bias=False)   # gate
        self.w2 = nn.Linear(hidden, n_embd, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

C_test = 768
gelu_ffn_w = 8 * C_test ** 2
swiglu = SwiGLU(C_test)
print(f"GPT-2 FFN Weight (8C²) : {gelu_ffn_w:>10,}")
print(f"SwiGLU Weight          : {count(swiglu):>10,}   (hidden = {swiglu.w1.out_features})")
print("출력 shape:", tuple(swiglu(torch.randn(2, 4, C_test)).shape))

mini GPT는 GPT-2 방식(GELU)으로 만들 거예요. SwiGLU는 "요즘은 이렇게 바뀌었다" 정도로 알아두면 충분합니다.

## 8. LLM과의 연결

Block 하나의 코드(Day 15~16에서 자세히)를 미리 보면, 오늘 만든 `MLP`가 이렇게 들어갑니다.

```python
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(...)   # Day 14
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp  = MLP(n_embd)                # 오늘!

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))        # 소통: Token끼리
        x = x + self.mlp(self.ln_2(x))         # 생각: Token마다
        return x
```

```text
(B, T, C) → Attention → (B, T, C) → MLP: (B, T, C) → (B, T, 4C) → (B, T, C)
```

## 9. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** nanoGPT 방식 `MLP`를 직접 완성하세요. (5절을 보지 않고 먼저 시도해보세요)
- 속성 이름: `c_fc`, `gelu`, `c_proj`, `dropout`
- 흐름: `c_fc → gelu → c_proj → dropout`

In [ ]:
class MyMLP(nn.Module):
    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        # TODO: 네 개의 Layer 만들기
        pass

    def forward(self, x):
        return None  # TODO

m = MyMLP(32)
assert hasattr(m, "c_fc") and hasattr(m, "c_proj"), "c_fc / c_proj 를 만들어주세요"
assert (m.c_fc.in_features, m.c_fc.out_features) == (32, 128), "c_fc의 크기를 다시 보세요"
assert (m.c_proj.in_features, m.c_proj.out_features) == (128, 32), "c_proj의 크기를 다시 보세요"
m.eval()
x = torch.randn(2, 6, 32)
out = m(x)
assert out is not None, "forward를 완성해주세요"
assert torch.allclose(out, m.c_proj(F.gelu(m.c_fc(x))), atol=1e-6), "흐름 순서를 다시 보세요"
print("Q1 정답!", tuple(out.shape))

**Q2.** FFN이 position-wise라는 걸 확인하는 함수를 완성하세요.
`(B, T, C)` 입력에서 **Token 벡터를 하나씩** FFN에 넣어 같은 shape의 결과를 만들어야 합니다.

In [ ]:
def ffn_token_by_token(ffn, x):
    B, T, C = x.shape
    out = None  # TODO: 반복문으로 하나씩 계산해서 (B, T, C) 결과 만들기
    return out

ffn = MLP(16)
x = torch.randn(3, 7, 16)
res = ffn_token_by_token(ffn, x)
assert res is not None, "함수를 완성해주세요"
assert res.shape == x.shape, "결과 shape가 입력과 같아야 해요"
assert torch.allclose(res, ffn(x), atol=1e-6), "통째로 넣은 결과와 달라요"
print("Q2 정답! FFN은 Token마다 따로 동작한다")

**Q3.** 파라미터 개수를 **손으로** 계산하세요. (bias 포함)
1. `MLP(64)` 하나의 파라미터 개수
2. GPT-2 small(`C = 768`)의 12층 전체 MLP 파라미터 개수

힌트: `nn.Linear(a, b)`의 파라미터는 `a × b + b` (Day 3)

In [ ]:
mlp_c64     = "?"   # TODO: 숫자
gpt2_mlp_12 = "?"   # TODO: 숫자

assert mlp_c64 == count(MLP(64)), "MLP(64)의 개수를 다시 세어보세요"
assert gpt2_mlp_12 == count(MLP(768)) * 12, "GPT-2 12층 전체를 다시 계산해보세요"
print("Q3 정답!")

**Q4.** 다음 중 **옳은** 설명을 하나 고르세요.

- A. FFN은 Token끼리 정보를 섞는다
- B. Attention은 각 Token을 다른 Token과 상관없이 독립적으로 처리한다
- C. FFN은 모든 Token에 같은 Weight를 쓰되, Token마다 따로 계산한다
- D. FFN의 중간(hidden) 크기는 항상 C와 같아야 한다

In [ ]:
import hashlib

answer = "?"   # TODO: "A", "B", "C", "D" 중 하나

assert answer in ("A", "B", "C", "D"), "A~D 중 하나를 골라주세요"
assert hashlib.sha256(answer.encode()).hexdigest() == "6b23c0d5f35d1b11f9b683f0b0a617355deb11277d91ae091d399c655b87940d", "2절 실험 결과를 다시 떠올려보세요"
print("Q4 정답!")

## 10. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. FFN은 Attention과 어떤 차이가 있는가? (커리큘럼 최종 질문)
2. "FFN은 position-wise다"라는 말은 무슨 뜻인가? 어떻게 확인할 수 있는가?
3. FFN은 왜 C → 4C로 늘렸다가 다시 C로 줄이는가?
4. Block 하나에서 FFN과 Attention 중 파라미터가 더 많은 쪽은? 대략 몇 배인가?
5. 생성할 때 `model.eval()`을 꼭 불러야 하는 이유는? (Dropout 관점)

**내 답:**

1.
2.
3.
4.
5.